In [17]:
# EEG Preprocessing Pipeline in Jupyter Notebook
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import mne
import pandas as pd


In [18]:
# CONFIGURATION
INPUT_SET = "C:/Users/sanja/Desktop/EEG_Project/EEG/eeg/sub-001/eeg/sub-001_task-eyesclosed_eeg.set"
OUTPUT_FIF = "sub-001_task-eyesclosed_preprocessed_raw_01.fif"
ORIG_PREVIEW = "orig_preview_CN.png"
CLEAN_PREVIEW = "clean_preview_CN.png"

if not os.path.exists(INPUT_SET):
    raise FileNotFoundError(f"{INPUT_SET} not found.")


In [19]:
# Load EEG .set file
raw = mne.io.read_raw_eeglab(INPUT_SET, preload=True)
print("Loaded:", INPUT_SET)
print("Channels:", raw.ch_names)

sfreq = raw.info['sfreq']

# Ensure EEG channel types
raw.set_channel_types({ch: 'eeg' for ch in raw.ch_names if ch.upper() not in ['EOG','ECG','EMG']})


Loaded: C:/Users/sanja/Desktop/EEG_Project/EEG/eeg/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
Channels: ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz']


<RawEEGLAB | sub-001_task-eyesclosed_eeg.set, 19 x 299900 (599.8 s), ~43.5 MiB, data loaded>

In [20]:
# 1) Re-reference to A1, A2 if available; else average reference
ref_chs = ['A1', 'A2']
present_refs = [ch for ch in ref_chs if ch in raw.ch_names]

if len(present_refs) == 2:
    raw.set_eeg_reference(ref_channels=present_refs)
    print(f"Re-referenced to average of {present_refs}")
else:
    raw.set_eeg_reference(ref_channels='average')
    print("A1/A2 not both present; used average reference")


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
A1/A2 not both present; used average reference


In [21]:
# 2) Apply 4th-order Butterworth band-pass (0.5 - 45 Hz)
raw.filter(l_freq=0.5, h_freq=45.0, method='iir', iir_params=dict(ftype='butter', order=4))
print("Applied 4th-order Butterworth band-pass filter 0.5-45 Hz")


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 0.50, 45.00 Hz: -6.02, -6.02 dB

Applied 4th-order Butterworth band-pass filter 0.5-45 Hz


In [22]:
# 3) Mark 0.5-second windows with std > 17
win_s = 0.5
win_samples = int(win_s * sfreq)
data = raw.get_data(picks='eeg')
nsamp = data.shape[1]
bad_intervals = []

for start in range(0, nsamp - win_samples + 1, win_samples):
    seg = data[:, start:start+win_samples]
    seg_std = seg.std(axis=1)
    if np.any(seg_std > 17.0):
        t0, t1 = start / sfreq, (start + win_samples) / sfreq
        bad_intervals.append((t0, t1))

# Merge overlapping intervals
merged = []
for s, e in bad_intervals:
    if not merged or s > merged[-1][1]:
        merged.append([s, e])
    else:
        merged[-1][1] = max(merged[-1][1], e)

ann = mne.Annotations(onset=[iv[0] for iv in merged],
                      duration=[iv[1]-iv[0] for iv in merged],
                      description=['BAD_std>17']*len(merged))
raw.set_annotations(raw.annotations + ann)
print(f"Annotated {len(merged)} bad {win_s}s windows (std > 17)")


Annotated 0 bad 0.5s windows (std > 17)


In [23]:
# 4) Artifact Subspace Reconstruction (ASR) or fallback repair
asr_applied = False
try:
    from mne.preprocessing import ASR
    asr = ASR(threshold=20.0, burst_criterion=0.5)
    raw = asr.fit_transform(raw)
    asr_applied = True
    print("ASR applied via mne.preprocessing.ASR")
except Exception:
    repaired = 0
    for onset, dur, desc in zip(raw.annotations.onset, raw.annotations.duration, raw.annotations.description):
        if 'BAD_std>17' in desc and dur <= 5.0:
            s0, s1 = int(onset * sfreq), int((onset+dur) * sfreq)
            for ch in range(data.shape[0]):
                sig = data[ch]
                left_val = sig[s0-1] if s0-1 >= 0 else sig[s1] if s1 < sig.size else 0.0
                right_val = sig[s1] if s1 < sig.size else sig[s0] if s0-1 >= 0 else 0.0
                sig[s0:s1] = np.linspace(left_val, right_val, s1-s0)
                data[ch] = sig
            repaired += 1
    if repaired > 0:
        raw._data[:data.shape[0], :] = data
        print(f"ASR not available. Interpolated {repaired} short bad segments (<=5 s)")
    else:
        print("ASR not available and no short bad segments to repair.")


ASR not available and no short bad segments to repair.


In [24]:
# 5) Run ICA with 19 components
n_components = min(19, len(raw.ch_names))
ica = mne.preprocessing.ICA(n_components=n_components, method='fastica', random_state=42, max_iter='auto')
ica.fit(raw, picks='eeg')
print(f"Fitted ICA with {n_components} components")


Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 19 components
Fitting ICA took 5.9s.
Fitted ICA with 19 components


In [25]:
# 6) Auto-detect artifact ICs
frontal_eye = [ch for ch in ['Fp1','Fp2'] if ch in raw.ch_names]
jaw_chs = [ch for ch in ['F7','F8','T7','T8'] if ch in raw.ch_names]

artifact_idxs = set()
sources = ica.get_sources(raw).get_data()

# Correlation with eye activity
if frontal_eye:
    proxy = raw.copy().pick_channels(frontal_eye).get_data().mean(axis=0)
    for ic in range(sources.shape[0]):
        corr = np.corrcoef(sources[ic], proxy)[0, 1]
        if abs(corr) > 0.4:
            artifact_idxs.add(ic)

# Correlation with jaw/muscle
if jaw_chs:
    proxy2 = raw.copy().pick_channels(jaw_chs).get_data().mean(axis=0)
    for ic in range(sources.shape[0]):
        corr = np.corrcoef(sources[ic], proxy2)[0, 1]
        if abs(corr) > 0.35:
            artifact_idxs.add(ic)

# Kurtosis heuristic fallback
if not artifact_idxs:
    kurt = pd.Series([pd.Series(sources[i]).kurtosis() for i in range(sources.shape[0])])
    high_kurt = kurt[abs(kurt) > 10].index.tolist()
    artifact_idxs.update(high_kurt)
    print(f"No proxies found; flagged by kurtosis: {high_kurt}")

artifact_idxs = sorted(list(artifact_idxs))
if artifact_idxs:
    ica.exclude = artifact_idxs
    raw_clean = ica.apply(raw.copy(), exclude=artifact_idxs)
    print(f"Removed ICA components: {artifact_idxs}")
else:
    raw_clean = raw.copy()
    print("No artifact ICA components identified automatically")


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Applying ICA to Raw instance
    Transforming to ICA space (19 components)
    Zeroing out 3 ICA components
    Projecting back using 19 PCA components
Removed ICA components: [0, 2, 8]


In [26]:
# 7) Save cleaned EEG data
raw_clean.save(OUTPUT_FIF, overwrite=True)
print("Saved cleaned data to:", OUTPUT_FIF)

# Save quick preview (first 5 s)
raw.plot(start=0.0, duration=5.0, n_channels=min(6, len(raw.ch_names)), show=False).savefig(ORIG_PREVIEW)
raw_clean.plot(start=0.0, duration=5.0, n_channels=min(6, len(raw_clean.ch_names)), show=False).savefig(CLEAN_PREVIEW)

print("Saved previews:", ORIG_PREVIEW, CLEAN_PREVIEW)


Writing C:\Users\sanja\sub-001_task-eyesclosed_preprocessed_raw_01.fif
Closing C:\Users\sanja\sub-001_task-eyesclosed_preprocessed_raw_01.fif
[done]
Saved cleaned data to: sub-001_task-eyesclosed_preprocessed_raw_01.fif
Saved previews: orig_preview_CN.png clean_preview_CN.png
